In [ ]:
!pip install ase sqsgenerator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.9 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import pandas as pd
from ase.build import bulk
from ase.data import atomic_numbers
from sqsgenerator import optimize, from_ase, write, parse_config
from sqsgenerator.core import ParseError
DIR='/content/drive'
HOME = os.path.join(DIR, 'MyDrive')
from google.colab import drive
drive.mount(DIR)
DIR = os.path.join(HOME,'Analyze7RHEA')
os.chdir(DIR)

Mounted at /content/drive


In [ ]:
def get_integer_counts(percentages: dict, total_atoms: int) -> dict:
    """
    Convert percentage concentrations to integer atom counts using Chemical Symbols.
    """
    counts = {}
    current_sum = 0

    # Use Chemical Symbols (str) as keys
    for sym, percent in percentages.items():
        count = int(round((percent / 100.0) * total_atoms))
        counts[sym] = count
        current_sum += count

    # Adjustment pass: Fix rounding errors
    diff = total_atoms - current_sum
    if diff != 0:
        most_abundant = max(counts, key=counts.get)
        counts[most_abundant] += diff
        print(f"Notice: Adjusted count of {most_abundant} by {diff} to match total atoms.")

    return {k: v for k, v in counts.items() if v > 0}

# ==========================================
# 1. Configuration
# ==========================================

df = pd.read_csv('RHEA7unique.csv')

# 2. Define strictly the columns you care about
interest_cols = ['Mo', 'Nb', 'Ta', 'Ti', 'V', 'W', 'Zr']

# 3. The Logic:
# First, use df[interest_cols] to ignore 'Formula' and 'Bulk' completely.
# Then convert to records and filter out zeros.
clean_dicts = [
    {element: value for element, value in row.items() if value != 0}
    for row in df[interest_cols].to_dict('records')
]

# 4. View the results
for i in range(len(clean_dicts)):
    composition_counts = clean_dicts[i]
    print(composition_counts)
    form = df.iloc[i]['Formula']
    print(form, (i+1))
    #break

    #composition_counts = {'Zr': 30, 'Ta': 23, 'Nb': 15, 'Ti': 15, 'V': 15, 'Mo': 15, 'W': 15}

    #composition_counts = {'Zr': 3.75, 'Ta': 2.875, 'Nb': 1.875, 'Ti': 1.875, 'V': 1.875, 'Mo': 1.875, 'W': 1.875}

    print(f"Target Composition: {composition_counts}")

    # ==========================================
    # 2. Generate Base 16-atom Structure
    # ==========================================
    # FIX: cubic=True gives the conventional cell (2 atoms).
    # Without this, it gives the primitive cell (1 atom).
    #prim = bulk('Ti', 'bcc', a=3.3, cubic=True)

    prim = bulk('Ti', 'bcc', a=3.3425, cubic=True)

    # 2 atoms/cell * 8 cells = 16 atoms
    base_atoms = prim.repeat((4, 4, 4))

    print(f"Base structure has {len(base_atoms)} sites.")

    # Convert to sqsgenerator structure object
    structure_input = from_ase(base_atoms)

    # Ensure species are passed as a list of strings
    species_list = structure_input.symbols

    # ==========================================
    # 3. Setup SQS Optimization
    # ==========================================
    config = {
        "structure": {
            "lattice": structure_input.lattice.tolist(),
            "coords": structure_input.frac_coords.tolist(),
            "species": species_list  # List of Strings ["Ti", "Ti", ...]
        },
        "composition": composition_counts, # Dict[str, int]
        "shell_weights": {1: 1.0, 2: 0.5, 3: 0.25},
        "iterations": 100_000,
        "iteration_mode": "random",
        "thread_config": [1]
    }

    print("Validating configuration...")
    parsed = parse_config(config)

    if isinstance(parsed, ParseError):
        print("\n!!! CONFIGURATION ERROR !!!")
        print(f"Message:   {parsed.msg}")
        print(f"Key:       {parsed.key}")
        print(f"Parameter: {parsed.parameter}")
        print(f"Code:      {parsed.code}")
        raise ValueError(f"Sqsgenerator configuration failed: {parsed.msg}")

    print("Configuration valid. Running optimization...")
    results = optimize(parsed)

    # ==========================================
    # 4. Get Best Result and Expand
    # ==========================================
    best_sqs_16 = results.best().structure()
    print(f"Best Objective: {results.best().objective}")
    write(best_sqs_16, os.path.join("RHEA7SQS2",f"{form}.poscar"))
    os.rename(os.path.join("RHEA7SQS2",f"{form}.poscar"),os.path.join("RHEA7SQS2",f"{form}"))

Streaming output truncated to the last 5000 lines.
Configuration valid. Running optimization...
Best Objective: 1.3435534084809448
{'Mo': 30, 'Ta': 15, 'V': 23, 'W': 15, 'Zr': 45}
Zr45Ta15V23Mo30W15 11299
Target Composition: {'Mo': 30, 'Ta': 15, 'V': 23, 'W': 15, 'Zr': 45}
Base structure has 128 sites.
Validating configuration...
Configuration valid. Running optimization...
Best Objective: 1.32792270531401
{'Mo': 15, 'Nb': 30, 'Ta': 15, 'Ti': 23, 'V': 15, 'Zr': 30}
Zr30Ta15Ti23Nb30V15Mo15 11300
Target Composition: {'Mo': 15, 'Nb': 30, 'Ta': 15, 'Ti': 23, 'V': 15, 'Zr': 30}
Base structure has 128 sites.
Validating configuration...
Configuration valid. Running optimization...
Best Objective: 3.2854750402576474
{'Mo': 30, 'Ti': 68, 'Zr': 30}
Zr30Ti68Mo30 11301
Target Composition: {'Mo': 30, 'Ti': 68, 'Zr': 30}
Base structure has 128 sites.
Validating configuration...
Configuration valid. Running optimization...
Best Objective: 0.08967320261437906
{'Ti': 30, 'V': 23, 'W': 75}
Ti30V23W75 11